In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sentence_transformers import SentenceTransformer

import faiss

In [2]:
PROCESSED_DATA_PATH = Path("../data/processed")

DEV_DATASET_PATH = (
    PROCESSED_DATA_PATH /
    "podcasts_subset_20k.csv"
)

print(DEV_DATASET_PATH)

..\data\processed\podcasts_subset_20k.csv


In [3]:
df = pd.read_csv(DEV_DATASET_PATH)

print(df.shape)

df.head()

(20000, 7)


,podcast_id,title,author,description,categories,semantic_density_score,combined_text
0,ff269332200b78cbabfe05ce6d2ab038,Chinwag Live Podcasts,Chinwag Live,Chinwag is an online community for digital pro...,"business, technology",4,Title: Chinwag Live Podcasts | Author: Chinwag...
1,b6e5d2929465dd2080be0bbced0c29cf,Agora Historia Oficial,Ágora Historia,Ágora Historia programa dedicado a la historia...,"society-culture, history",4,Title: Agora Historia Oficial | Author: Ágora ...
2,95a7ebc3e48c025d957091577b9c4c64,Einstellungssache,emplify,"Alles rund um Employer Branding, Personalmarke...","business-careers, business",4,Title: Einstellungssache | Author: emplify | C...
3,d49b1eba62b6d34a8bea0f7021946c96,Schon wieder nur vom Wetter geredet...,Leopold-Franzens-Universität Innsbruck,Mathias Rotach verfolgt einen Ansatz des Zusam...,"education-courses, education, science",4,Title: Schon wieder nur vom Wetter geredet... ...
4,4ee563e24c71b99607bf035e6765d727,Le Dossier du Jour France Bleu Cotentin,France Bleu,Mieux vivre au quotidien grâce aux conseils de...,business,4,Title: Le Dossier du Jour France Bleu Cotentin...


In [4]:
print(df.columns)

Index(['podcast_id', 'title', 'author', 'description', 'categories',
       'semantic_density_score', 'combined_text'],
      dtype='object')


In [5]:
samples = df["combined_text"].sample(
    5,
    random_state=42
)

for i, text in enumerate(samples, 1):
    
    print(f"\n========== SAMPLE {i} ==========\n")
    
    print(text[:1000])


========== SAMPLE 1 ==========

Title: Terceira Terra | Author: Terceira Terra | Categories: arts-books, arts, fiction, fiction-science-fiction | Description: Todo dia, um show sobre video games, quadrinhos, ficção científica, Star Wars, séries de TV, RPG e muito mais. Se é geek, aqui há.

========== SAMPLE 2 ==========

Title: Made in Sweden: the podcast of The Father | Author: Anton Svensson | Categories: arts-books, arts, society-culture | Description: The Made in Sweden podcast is the true story behind the novel The Father by Anton Svensson. This is an unforgettable, thrilling real-life story of how three brothers grew up to become Sweden’s most wanted criminals, as told by fourth brother Stefan Thunberg and crime journalist-turned-author Anders Roslund. There are 6 episodes released every Friday for 6 weeks.The Father is available to buy from iTunes as an ebook and digital audiobook. The hardback is available from all good bookstores. For details, go to www.thecrimevault.com/eboo

In [6]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [7]:
sample_texts = (
    df["combined_text"]
    .head(5)
    .tolist()
)

test_embeddings = model.encode(
    sample_texts
)

print(type(test_embeddings))

print(test_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    test_embeddings
)

print(similarity_matrix)

[[ 1.          0.19248402  0.14655603 -0.07135014  0.0513371 ]
 [ 0.19248402  1.0000001   0.23904139  0.06330445  0.321742  ]
 [ 0.14655603  0.23904139  1.0000002   0.0903005   0.4199649 ]
 [-0.07135014  0.06330445  0.0903005   0.9999999   0.1292806 ]
 [ 0.0513371   0.321742    0.4199649   0.1292806   1.0000002 ]]


## SECTION — FULL EMBEDDING GENERATION

We now generate embeddings for the entire 20k subset. 

### Why Batching?
Processing 20,000 records one by one would be inefficient. We use batching to leverage vectorized operations (and GPU acceleration if available).

### Normalization
We set `normalize_embeddings=True`. This scales every vector to a length of 1. In a unit-normalized hypersphere, the **Inner Product** between two vectors is identical to their **Cosine Similarity**. This makes downstream retrieval faster and simpler.

In [9]:
print(f"Generating embeddings for {len(df)} podcasts...")

embeddings = model.encode(
    df["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Embedding generation complete.")
print(f"Shape: {embeddings.shape}")
print(f"Dimensions: {embeddings.shape[1]}")

Generating embeddings for 20000 podcasts...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embedding generation complete.
Shape: (20000, 384)
Dimensions: 384


## SECTION — SAVE EMBEDDINGS

Generating embeddings is computationally expensive. We persist them to disk to avoid re-calculating them in future sessions.

In [10]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

EMBEDDINGS_PATH = ARTIFACTS_DIR / "podcast_embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)
print(f"Embeddings saved to {EMBEDDINGS_PATH}")

Embeddings saved to ..\artifacts\podcast_embeddings.npy


## SECTION — BUILD FAISS INDEX

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. 

### Why FAISS?
Standard exhaustive search (comparing a query against every record) becomes slow as datasets grow. FAISS provides optimized algorithms to find the nearest neighbors in high-dimensional space.

### IndexFlatIP
We use `IndexFlatIP` (Inner Product). Since our embeddings are normalized, the inner product equals cosine similarity. `Flat` means it's an exhaustive (brute-force) index, which is perfectly fast for 20k-100k records and guarantees 100% accuracy.

In [11]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add embeddings to the index
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")

FAISS index built with 20000 vectors.


## SECTION — SAVE FAISS INDEX

Persistence allows us to load the index instantly in production without rebuilding it from the raw embeddings.

In [12]:
INDEX_PATH = ARTIFACTS_DIR / "faiss_index.index"
faiss.write_index(index, str(INDEX_PATH))
print(f"FAISS index saved to {INDEX_PATH}")

FAISS index saved to ..\artifacts\faiss_index.index


## SECTION — BUILD PODCAST ID MAPPING

FAISS only returns the *integer index* of the nearest vectors. We need a mapping to translate these integers back into podcast metadata (IDs, Titles, etc.).

In [13]:
import pickle

# We store a list of metadata dictionaries where the list index matches the FAISS index
metadata_mapping = df[['podcast_id', 'title', 'categories', 'description']].to_dict('records')

METADATA_PATH = ARTIFACTS_DIR / "podcast_metadata.pkl"
with open(METADATA_PATH, 'wb') as f:
    pickle.dump(metadata_mapping, f)

print(f"Metadata mapping saved to {METADATA_PATH}")

Metadata mapping saved to ..\artifacts\podcast_metadata.pkl


## SECTION — SEMANTIC SEARCH FUNCTION

A reusable function to query the engine.

In [14]:
def get_similar_podcasts(query, top_k=5):
    # 1. Embed query
    query_vector = model.encode([query], normalize_embeddings=True)
    
    # 2. Search FAISS index
    # distances: similarity scores (since we used IndexFlatIP and normalized vectors)
    # indices: the positions in our metadata_mapping
    distances, indices = index.search(query_vector, top_k)
    
    results = []
    for i in range(top_k):
        idx = indices[0][i]
        score = distances[0][i]
        
        item = metadata_mapping[idx].copy()
        item['similarity_score'] = float(score)
        results.append(item)
        
    return pd.DataFrame(results)

print("Search function ready.")

Search function ready.


## SECTION — RETRIEVAL VALIDATION

Let's test the engine with diverse queries to verify semantic understanding.

In [15]:
queries = [
    "Artificial Intelligence and future of technology",
    "True crime stories about unsolved mysteries",
    "How to improve mental health and anxiety",
    "Stand-up comedy and funny stories"
]

for q in queries:
    print(f"\n{'='*20} QUERY: {q} {'='*20}")
    results = get_similar_podcasts(q, top_k=3)
    display(results[['title', 'categories', 'similarity_score']])


==================== QUERY: Artificial Intelligence and future of technology ====================


,title,categories,similarity_score
0,A.I. Meets World,"business, technology, business-entrepreneurship",0.442237
1,Brain Inspired,"technology, science, science-natural-sciences",0.439862
2,SMART Stories for Smart People,"society-culture, technology",0.437228



==================== QUERY: True crime stories about unsolved mysteries ====================


,title,categories,similarity_score
0,Once Upon A Crime | True Crime,"true-crime, history",0.595295
1,Crime Stories with Nancy Grace,"news, true-crime",0.549611
2,Ann Arbor Stories | Ann Arbor District Library,"arts, government",0.538673



==================== QUERY: How to improve mental health and anxiety ====================


,title,categories,similarity_score
0,Anxious Ramblings,health-fitness,0.477289
1,Affirmations dealing with anxiety with the pow...,NaN,0.474335
2,Anxiety Busterz,"christianity, religion-spirituality",0.446402



==================== QUERY: Stand-up comedy and funny stories ====================


,title,categories,similarity_score
0,YEAH...BOUT THAT COMEDY PODCAST,comedy,0.585846
1,Off The Rails Podcast: or (The Unexpected Humo...,"society-culture, tv-film, comedy",0.561111
2,Nights Of Our Lives,"comedy, arts, arts-performing-arts",0.542150
